<a href="https://colab.research.google.com/github/thasleenava/malformer-x/blob/main/Malware_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install all required packages
!pip install -q torch-geometric torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
!pip install -q imbalanced-learn shap lime scikit-learn pandas numpy matplotlib seaborn
!pip install -q captum pyod statsmodels scipy kaggle

import subprocess, sys
print('✅ All packages installed')

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 423, in run
    _, build_failures = build(
                        ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/wheel_builder.py", line 319, in build
    wheel_file = _build_one(
                 ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/wheel_builder.py", line 193, in _build_one
    wheel_path = _build_one_inside_env(
                 ^^^^^^^^^^^^^

In [ ]:
import os, time, warnings, random, json, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')

# ── Sklearn ───────────────────────────────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    confusion_matrix, classification_report, cohen_kappa_score
)
from sklearn.feature_selection import mutual_info_classif
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# ── PyG ───────────────────────────────────────────────────────────────────────
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoLoader
from torch_geometric.nn import (
    GATConv, GINConv, global_mean_pool, global_max_pool,
    BatchNorm, TransformerConv
)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | PyTorch: {torch.__version__}')

DATA_DIR = '/content/data'
OUT_DIR  = '/content/outputs'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)

In [ ]:
from google.colab import files

uploaded = files.upload()  # upload your CSV

if uploaded:
    uploaded_filename = next(iter(uploaded))

    src_path = os.path.join('/content', uploaded_filename)
    dst_path = CSV_PATH

    os.rename(src_path, dst_path)

    print(f"✅ File moved to: {dst_path}")

In [ ]:
import pandas as pd

if os.path.exists(CSV_PATH):
    print("✅ Loading REAL dataset...")
    df_raw = pd.read_csv(CSV_PATH)
else:
    raise FileNotFoundError("❌ CSV not found. Upload failed.")

print("Shape:", df_raw.shape)
print(df_raw.head(3))

In [ ]:
FAMILY_COL = 'Category'
LABEL_COL  = 'Class'   # NOT 'Label'

In [ ]:
df_raw.rename(columns={
    'Category': 'Family',
    'Class': 'Label'
}, inplace=True)

In [ ]:
def extract_superclass(family_name):
    name = str(family_name).lower()

    if 'benign' in name:
        return 'Benign'
    elif 'ransomware' in name:
        return 'Ransomware'
    elif 'spyware' in name:
        return 'Spyware'
    elif 'trojan' in name:
        return 'Trojan'
    else:
        return 'Trojan'   # fallback

In [ ]:
df_raw['SuperClass'] = df_raw['Family'].apply(extract_superclass)

In [ ]:
print(df_raw['SuperClass'].value_counts())
print(df_raw['SuperClass'].unique())

In [ ]:
# ─── 2.1 Clean ───────────────────────────────────────────────────────────────
df = df_raw.copy()
df[FEAT_COLS] = df[FEAT_COLS].apply(pd.to_numeric, errors='coerce')
df[FEAT_COLS] = df[FEAT_COLS].fillna(df[FEAT_COLS].median())
df = df.drop_duplicates().reset_index(drop=True)
print(f'After cleaning: {df.shape}')

# Ensure 'SuperClass' is present before encoding
if 'SuperClass' not in df.columns:
    # Assuming extract_superclass and FAMILY_TO_SUPER are defined in the environment
    df['SuperClass'] = df['Family'].apply(extract_superclass)

# Define a mapping function to convert granular family names to one of the 15 FAMILY_CLASSES
def map_to_defined_family_class(granular_name):
    name_lower = str(granular_name).lower()
    if 'benign' in name_lower:
        return 'Benign'
    elif 'wannacry' in name_lower:
        return 'WannaCry'
    elif 'cerber' in name_lower:
        return 'Cerber'
    elif 'cryptolocker' in name_lower:
        return 'CryptoLocker'
    elif 'locky' in name_lower:
        return 'Locky'
    elif 'teslacrypt' in name_lower:
        return 'TeslaCrypt'
    elif 'emotet' in name_lower:
        return 'Emotet'
    elif 'agent' in name_lower:
        return 'Agent'
    elif 'dridex' in name_lower:
        return 'Dridex'
    elif 'kovter' in name_lower:
        return 'Kovter'
    elif 'netwire' in name_lower:
        return 'NetWire'
    elif 'darkcomet' in name_lower:
        return 'DarkComet'
    elif 'xtreme' in name_lower:
        return 'Xtreme'
    elif 'njrat' in name_lower:
        return 'NjRAT'
    elif 'remcosrat' in name_lower:
        return 'RemcosRAT'
    # Fallback for ransomware, trojan, spyware if specific names not matched
    elif 'ransomware' in name_lower:
        return 'WannaCry' # Arbitrary choice from FAMILY_CLASSES for ransomware
    elif 'trojan' in name_lower:
        return 'Emotet'   # Arbitrary choice from FAMILY_CLASSES for trojan
    elif 'spyware' in name_lower:
        return 'DarkComet' # Arbitrary choice from FAMILY_CLASSES for spyware
    return 'Benign' # Default fallback if no match

# Apply the mapping to the 'Family' column
df['Family_mapped'] = df['Family'].apply(map_to_defined_family_class)

# ─── 2.2 Encode labels ───────────────────────────────────────────────────────
le_super  = LabelEncoder().fit(SUPER_CLASSES)
le_family = LabelEncoder().fit(FAMILY_CLASSES)

df['super_enc']  = le_super.transform(df['SuperClass'])
df['family_enc'] = le_family.transform(df['Family_mapped'])

# Encode binary 'Label' column (Benign vs Malware)
le_binary = LabelEncoder()
df['binary_enc'] = le_binary.fit_transform(df['Label'])
y_binary = df['binary_enc'].values
y_super  = df['super_enc'].values     # 4-class
y_family = df['family_enc'].values    # 15-class
N_SUPER  = len(SUPER_CLASSES)         # 4
N_FAM    = len(FAMILY_CLASSES)        # 15

print(f'Super distribution:  {np.bincount(y_super)}')
print(f'Family distribution: {np.bincount(y_family)}')

# ─── 2.3 Normalisation: Log1p + RobustScaler ─────────────────────────────────
X_raw   = df[FEAT_COLS].values.astype(np.float32)
X_log   = np.log1p(np.clip(X_raw, 0, None))
scaler  = RobustScaler()
X_sc    = scaler.fit_transform(X_log).astype(np.float32)

# ─── 2.4 Mutual-Info Feature Selection (top 45) ───────────────────────────────
K = min(45, X_sc.shape[1])
mi = mutual_info_classif(X_sc, y_binary, random_state=SEED)
top_idx = np.sort(np.argsort(mi)[::-1][:K])
X_sel   = X_sc[:, top_idx]
sel_names = [FEAT_COLS[i] for i in top_idx]
N_FEAT  = X_sel.shape[1]
print(f'Selected {N_FEAT} features')

# ─── 2.5 Hold-out test set (15%) ─────────────────────────────────────────────
# 5-Fold CV runs on the 85% pool; final test is a locked-away 15%
idx = np.arange(len(X_sel))
idx_pool, idx_test = train_test_split(idx, test_size=0.15,
                                       stratify=y_super, random_state=SEED)
X_pool  = X_sel[idx_pool];  y_sup_pool = y_super[idx_pool]
y_bin_pool = y_binary[idx_pool]; y_fam_pool = y_family[idx_pool]
X_test  = X_sel[idx_test];  y_sup_test = y_super[idx_test]
y_bin_test = y_binary[idx_test]; y_fam_test = y_family[idx_test]
print(f'Pool: {X_pool.shape} | Test (locked): {X_test.shape}')

In [ ]:
# ─── Node groups (process / DLL / handle / API hook) ─────────────────────────
q = N_FEAT // 4
GROUP_SLICES = [slice(0,q), slice(q,2*q), slice(2*q,3*q), slice(3*q,N_FEAT)]
N_GROUPS = len(GROUP_SLICES)

edge_index_base = torch.tensor(
    [[i,j] for i in range(N_GROUPS) for j in range(N_GROUPS) if i!=j],
    dtype=torch.long).t().contiguous()

NODE_STAT_DIM = 8  # per-group statistics

def sample_to_graph(x_row, y_bin, y_sup, y_fam):
    node_feats = []
    for sl in GROUP_SLICES:
        g = x_row[sl]
        stats = np.array([
            g.mean(), g.std(), g.max(), g.min(),
            g.sum(), (g>0).mean(),
            np.percentile(g,25), np.percentile(g,75)
        ], dtype=np.float32)
        node_feats.append(stats)
    nf = np.stack(node_feats)          # [N_GROUPS, NODE_STAT_DIM]

    edge_attr = []
    for s,t in edge_index_base.t().tolist():
        vs,vt = nf[s], nf[t]
        cos = float((vs@vt)/(np.linalg.norm(vs)*np.linalg.norm(vt)+1e-8))
        edge_attr.append([cos])

    return Data(
        x         = torch.tensor(nf,         dtype=torch.float),
        edge_index= edge_index_base,
        edge_attr = torch.tensor(edge_attr,  dtype=torch.float),
        raw_feat  = torch.tensor(x_row,      dtype=torch.float).unsqueeze(0),
        y_bin     = torch.tensor([y_bin],    dtype=torch.long),
        y_sup     = torch.tensor([y_sup],    dtype=torch.long),
        y_fam     = torch.tensor([y_fam],    dtype=torch.long),
    )

def build_dataset(X, y_bin, y_sup, y_fam, tag=''):
    g = [sample_to_graph(X[i],int(y_bin[i]),int(y_sup[i]),int(y_fam[i])) for i in range(len(X))]
    if tag: print(f'{tag}: {len(g)} graphs')
    return g

BATCH = 256
test_graphs  = build_dataset(X_test, y_bin_test, y_sup_test, y_fam_test, 'Test (locked)')
test_loader  = GeoLoader(test_graphs, batch_size=BATCH, shuffle=False, num_workers=0)
print(f'NODE_STAT_DIM={NODE_STAT_DIM}  N_FEAT={N_FEAT}')

In [ ]:
# ─── Anti-overfitting measures built into the model ──────────────────────────
# 1. DropPath (stochastic depth) in Transformer layers
# 2. Higher dropout (0.3 in heads, 0.2 in encoder)
# 3. Label smoothing in Focal Loss
# 4. Spectral norm on linear layers option
# 5. Smaller hidden dim (96) to reduce capacity

HIDDEN  = 96
N_HEADS = 4
DROP    = 0.25


class GATEncoder(nn.Module):
    def __init__(self, in_dim, hidden, heads=N_HEADS, edge_dim=1):
        super().__init__()
        self.conv1 = TransformerConv(in_dim, hidden//heads, heads=heads,
                                      edge_dim=edge_dim, dropout=DROP)
        self.bn1   = BatchNorm(hidden)
        self.conv2 = TransformerConv(hidden, hidden//heads, heads=heads,
                                      edge_dim=edge_dim, dropout=DROP)
        self.bn2   = BatchNorm(hidden)
        self.drop  = nn.Dropout(DROP)

    def forward(self, x, edge_index, edge_attr, batch):
        x = self.drop(F.elu(self.bn1(self.conv1(x, edge_index, edge_attr))))
        x = self.drop(F.elu(self.bn2(self.conv2(x, edge_index, edge_attr))))
        return x


class GINSubgraph(nn.Module):
    def __init__(self, in_dim, hidden):
        super().__init__()
        mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden),
            nn.ReLU(), nn.Dropout(DROP), nn.Linear(hidden, hidden)
        )
        self.conv = GINConv(mlp, train_eps=True)
        self.bn   = BatchNorm(hidden)

    def forward(self, x, edge_index):
        return F.relu(self.bn(self.conv(x, edge_index)))


class GraphReadout(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(hidden*2, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(DROP)
        )
    def forward(self, x, batch):
        return self.proj(torch.cat([global_mean_pool(x,batch),
                                     global_max_pool(x,batch)], dim=-1))


class RawTransformerEncoder(nn.Module):
    def __init__(self, raw_dim, hidden, n_layers=2):
        super().__init__()
        self.tok    = nn.Linear(1, hidden)
        self.pos    = nn.Embedding(raw_dim+1, hidden)  # +1 for CLS
        enc_layer   = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=N_HEADS, dim_feedforward=hidden*2,
            dropout=DROP, activation='gelu', batch_first=True, norm_first=True
        )
        self.trans  = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.cls    = nn.Parameter(torch.randn(1,1,hidden)*0.02)
        self.norm   = nn.LayerNorm(hidden)
        # Store attention weights for XAI
        self.attn_weights = None

    def forward(self, x):
        B, D = x.shape
        tok   = self.tok(x.unsqueeze(-1))                     # [B,D,H]
        pos   = self.pos(torch.arange(D, device=x.device))    # [D,H]
        tok   = tok + pos
        cls   = self.cls.expand(B,-1,-1)                      # [B,1,H]
        seq   = torch.cat([cls, tok], dim=1)                  # [B,D+1,H]
        out   = self.trans(seq)
        return self.norm(out[:,0,:])


class CrossAttentionFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, N_HEADS, dropout=DROP, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.proj = nn.Sequential(
            nn.Linear(dim*2, dim), nn.LayerNorm(dim),
            nn.GELU(), nn.Dropout(DROP)
        )
    def forward(self, g, s):
        a, _ = self.attn(g.unsqueeze(1), s.unsqueeze(1), s.unsqueeze(1))
        a     = self.norm(a.squeeze(1) + g)
        return self.proj(torch.cat([a, s], dim=-1))


class ClassHead(nn.Module):
    def __init__(self, hidden, n_cls):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden, hidden//2), nn.LayerNorm(hidden//2),
            nn.GELU(), nn.Dropout(0.3),
            nn.Linear(hidden//2, n_cls)
        )
    def forward(self, x): return self.net(x)


class MalFormerX(nn.Module):
    """
    Three-head: binary, 4-super-class, 15-family.
    Hierarchical shared encoder with split family head.
    """
    def __init__(self, node_dim, raw_dim, hidden=HIDDEN,
                 n_super=4, n_family=15):
        super().__init__()
        self.gat     = GATEncoder(node_dim, hidden)
        self.gin     = GINSubgraph(hidden, hidden)
        self.readout = GraphReadout(hidden)
        self.trans   = RawTransformerEncoder(raw_dim, hidden)
        self.fusion  = CrossAttentionFusion(hidden)
        # Three task heads
        self.h_bin   = ClassHead(hidden, 2)
        self.h_sup   = ClassHead(hidden, n_super)
        # Family head: hidden + super_logits concatenated → richer context
        self.h_fam   = nn.Sequential(
            nn.Linear(hidden + n_super, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(0.3),
            nn.Linear(hidden, n_family)
        )

    def forward(self, batch):
        h_g  = self.gat(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        h_g  = self.gin(h_g, batch.edge_index)
        h_g  = self.readout(h_g, batch.batch)
        raw  = batch.raw_feat.view(batch.num_graphs, -1)
        h_s  = self.trans(raw)
        fused = self.fusion(h_g, h_s)
        o_bin = self.h_bin(fused)
        o_sup = self.h_sup(fused)
        # Hierarchical family: fused + softmax(super) as context
        sup_soft = F.softmax(o_sup.detach(), dim=-1)
        o_fam = self.h_fam(torch.cat([fused, sup_soft], dim=-1))
        return o_bin, o_sup, o_fam


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

# Quick instantiation to verify shapes
_demo = MalFormerX(NODE_STAT_DIM, N_FEAT, hidden=HIDDEN, n_super=N_SUPER, n_family=N_FAM)
print(_demo)
print(f'\n✅ Parameters: {count_params(_demo):,}')
del _demo

In [ ]:
N_FOLDS    = 5
N_EPOCHS   = 60
PATIENCE   = 10
LR         = 2e-4
WD         = 5e-4   # weight decay — key anti-overfitting

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Storage
fold_metrics_sup = []   # 4-class per-fold
fold_metrics_fam = []   # 15-class per-fold
fold_metrics_bin = []   # binary per-fold
best_fold_models = []


def train_epoch(model, loader, opt, crit_bin, crit_sup, crit_fam):
    model.train()
    total = 0
    for batch in loader:
        batch = batch.to(DEVICE)
        # Mixup
        batch, perm, lam = mixup_graphs(batch, MIXUP_ALPHA)
        opt.zero_grad()
        o_bin, o_sup, o_fam = model(batch)
        y_b = batch.y_bin.squeeze(); y_s = batch.y_sup.squeeze(); y_f = batch.y_fam.squeeze()
        if perm is not None:
            l = (L_BIN * mixup_loss(crit_bin, o_bin, y_b, y_b[perm], lam) +
                 L_SUP * mixup_loss(crit_sup, o_sup, y_s, y_s[perm], lam) +
                 L_FAM * mixup_loss(crit_fam, o_fam, y_f, y_f[perm], lam))
        else:
            l = (L_BIN*crit_bin(o_bin,y_b) + L_SUP*crit_sup(o_sup,y_s)
                 + L_FAM*crit_fam(o_fam,y_f))
        l.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += l.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    bins_l, sups_l, fams_l = [], [], []
    yb_l,   ys_l,  yf_l   = [], [], []
    for batch in loader:
        batch = batch.to(DEVICE)
        o_bin, o_sup, o_fam = model(batch)
        bins_l.append(F.softmax(o_bin,-1).cpu())
        sups_l.append(F.softmax(o_sup,-1).cpu())
        fams_l.append(F.softmax(o_fam,-1).cpu())
        yb_l.append(batch.y_bin.cpu()); ys_l.append(batch.y_sup.cpu())
        yf_l.append(batch.y_fam.cpu())
    bin_prob = torch.cat(bins_l).numpy()
    sup_prob = torch.cat(sups_l).numpy()
    fam_prob = torch.cat(fams_l).numpy()
    y_b = torch.cat(yb_l).squeeze().numpy()
    y_s = torch.cat(ys_l).squeeze().numpy()
    y_f = torch.cat(yf_l).squeeze().numpy()
    return dict(
        y_bin=y_b, bin_prob=bin_prob[:,1], bin_pred=bin_prob.argmax(1),
        y_sup=y_s, sup_prob=sup_prob,      sup_pred=sup_prob.argmax(1),
        y_fam=y_f, fam_prob=fam_prob,      fam_pred=fam_prob.argmax(1),
    )


def compute_binary_metrics(r):
    y,p,pr = r['y_bin'], r['bin_pred'], r['bin_prob']
    TP=np.sum((p==1)&(y==1)); TN=np.sum((p==0)&(y==0))
    FP=np.sum((p==1)&(y==0)); FN=np.sum((p==0)&(y==1))
    acc=( TP+TN)/(TP+TN+FP+FN+1e-9)
    prec=TP/(TP+FP+1e-9); rec=TP/(TP+FN+1e-9)
    f1=2*prec*rec/(prec+rec+1e-9)
    try:    auc=roc_auc_score(y,pr)
    except: auc=0
    try:    pr_auc=average_precision_score(y,pr)
    except: pr_auc=0
    return dict(acc=acc,prec=prec,rec=rec,f1=f1,auc=auc,pr_auc=pr_auc,
                mcc=matthews_corrcoef(y,p),
                fpr=FP/(FP+TN+1e-9),fnr=FN/(FN+TP+1e-9))


def compute_multi_metrics(y_true, y_pred, y_prob, class_names):
    f1m  = f1_score(y_true,y_pred,average='macro',  zero_division=0)
    f1w  = f1_score(y_true,y_pred,average='weighted',zero_division=0)
    acc  = accuracy_score(y_true,y_pred)
    kap  = cohen_kappa_score(y_true,y_pred)
    try:    auc=roc_auc_score(label_binarize(y_true,classes=np.arange(len(class_names))),
                               y_prob,average='macro',multi_class='ovr')
    except: auc=0
    return dict(f1_macro=f1m,f1_weighted=f1w,accuracy=acc,kappa=kap,auc_ovr=auc)


print(f'Starting {N_FOLDS}-Fold CV on pool ({len(X_pool)} samples)...')
t_cv_start = time.time()

for fold, (tr_i, va_i) in enumerate(skf.split(X_pool, y_sup_pool)):
    print(f'\n{'='*55}')
    print(f' FOLD {fold+1}/{N_FOLDS}')
    print(f'{'='*55}')

    X_tr, X_va = X_pool[tr_i], X_pool[va_i]
    y_b_tr,y_b_va = y_bin_pool[tr_i],  y_bin_pool[va_i]
    y_s_tr,y_s_va = y_sup_pool[tr_i],  y_sup_pool[va_i]
    y_f_tr,y_f_va = y_fam_pool[tr_i],  y_fam_pool[va_i]

    # SMOTE per-fold (on super labels for stratification)
    try:
        sm = SMOTE(random_state=SEED+fold, k_neighbors=3)
        X_tr_sm, y_s_sm = sm.fit_resample(X_tr, y_s_tr)
        # Reindex family and binary labels after SMOTE
        # (SMOTE creates synthetic; approximate by nearest original)
        from sklearn.neighbors import KNeighborsClassifier
        knn_f = KNeighborsClassifier(n_neighbors=1).fit(X_tr, y_f_tr)
        knn_b = KNeighborsClassifier(n_neighbors=1).fit(X_tr, y_b_tr)
        y_f_sm = knn_f.predict(X_tr_sm)
        y_b_sm = knn_b.predict(X_tr_sm)
    except Exception as e:
        print(f'  SMOTE failed ({e}), using original')
        X_tr_sm,y_s_sm,y_f_sm,y_b_sm = X_tr,y_s_tr,y_f_tr,y_b_tr

    # Per-fold class weights
    cw_sup = torch.tensor(
        compute_class_weight('balanced',classes=np.unique(y_s_sm),y=y_s_sm),
        dtype=torch.float).to(DEVICE)
    cw_fam = torch.tensor(
        compute_class_weight('balanced',classes=np.arange(N_FAM),y=y_f_sm),
        dtype=torch.float).to(DEVICE)

    crit_bin = FocalLoss(GAMMA_FOCAL, smoothing=LABEL_SMOOTH)
    crit_sup = FocalLoss(GAMMA_FOCAL, weight=cw_sup, smoothing=LABEL_SMOOTH)
    crit_fam = FocalLoss(GAMMA_FOCAL, weight=cw_fam, smoothing=LABEL_SMOOTH)

    # Build fold loaders
    tr_graphs = build_dataset(X_tr_sm, y_b_sm, y_s_sm, y_f_sm)
    va_graphs = build_dataset(X_va,    y_b_va, y_s_va, y_f_va)
    tr_loader = GeoLoader(tr_graphs, batch_size=BATCH, shuffle=True,  num_workers=0)
    va_loader = GeoLoader(va_graphs, batch_size=BATCH, shuffle=False, num_workers=0)

    # Fresh model each fold
    model = MalFormerX(NODE_STAT_DIM, N_FEAT, HIDDEN, N_SUPER, N_FAM).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = CosineAnnealingWarmRestarts(opt, T_0=20, T_mult=2, eta_min=1e-6)

    best_val, pat, best_state = 0.0, 0, None
    tr_losses, va_f1s = [], []

    for ep in range(1, N_EPOCHS+1):
        tl = train_epoch(model, tr_loader, opt, crit_bin, crit_sup, crit_fam)
        rv = evaluate(model, va_loader)
        vf = f1_score(rv['y_sup'], rv['sup_pred'], average='macro', zero_division=0)
        sched.step(ep)
        tr_losses.append(tl); va_f1s.append(vf)

        if vf > best_val:
            best_val = vf; pat = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            pat += 1
            if pat >= PATIENCE:
                print(f'  Early stop @ epoch {ep}')
                break
        if ep % 10 == 0:
            vf_fam = f1_score(rv['y_fam'],rv['fam_pred'],average='macro',zero_division=0)
            print(f'  Ep{ep:3d} | loss={tl:.4f} | val F1-sup={vf:.4f} | '
                  f'val F1-fam={vf_fam:.4f} | best={best_val:.4f}')

    # Evaluate best model on val
    model.load_state_dict(best_state)
    rv = evaluate(model, va_loader)

    bm = compute_binary_metrics(rv)
    sm = compute_multi_metrics(rv['y_sup'],rv['sup_pred'],rv['sup_prob'],SUPER_CLASSES)
    fm = compute_multi_metrics(rv['y_fam'],rv['fam_pred'],rv['fam_prob'],FAMILY_CLASSES)

    fold_metrics_bin.append(bm)
    fold_metrics_sup.append(sm)
    fold_metrics_fam.append(fm)
    best_fold_models.append(copy.deepcopy(model.state_dict()))

    print(f'  Fold {fold+1} → Binary F1={bm["f1"]:.4f} | '
          f'Super F1-mac={sm["f1_macro"]:.4f} | '
          f'Family F1-mac={fm["f1_macro"]:.4f}')

    # Plot fold learning curves
    fig,ax=plt.subplots(figsize=(8,3))
    ax.plot(tr_losses,label='Train Loss'); ax.plot(va_f1s,label='Val F1-sup')
    ax.set_title(f'Fold {fold+1} Learning Curves'); ax.legend()
    ax.set_xlabel('Epoch')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/fold{fold+1}_curves.png',dpi=120)
    plt.show()

TOTAL_CV_TIME = time.time() - t_cv_start
print(f'\n✅ CV complete in {TOTAL_CV_TIME/3600:.4f} CPU-hours')
